# Интерактивный селектор материалов по СП 63.13330.2018

Настоящий блокнот предназначен для оперативного выбора и анализа расчетных и нормативных характеристик бетона и арматуры в соответствии с требованиями **СП 63.13330.2018** *«Бетонные и железобетонные конструкции. Основные положения»* (с Изменениями № 1).

### Возможности инструмента:
1. **Нормативные и расчетные сопротивления** для I и II групп предельных состояний ($R_b, R_{bt}, R_{bn}, R_{btn}, R_s, R_{sc}, R_{sn}$).
2. **Учет коэффициентов условий работы** $\gamma_{b1} \dots \gamma_{b5}$ (длительность нагрузки, вертикальное бетонирование и др.).
3. **Деформационные характеристики и ползучесть**: начальный $E_b$ и приведенный $E_{b,red}$ модули, коэффициент ползучести $\varphi_{b,cr}$, предельные деформации $\varepsilon_{b0}, \varepsilon_{b2}, \varepsilon_{s0}, \varepsilon_{s2}$.
4. **Построение деформационных диаграмм состояния**: двухлинейные (п. 6.1.21, 6.2.14), трехлинейные (п. 6.1.20) и криволинейные (Приложение Г).
5. **Экспорт готового Python-кода** для интеграции в последующие расчетные блокноты.

In [1]:
import sys
from pathlib import Path

# Обеспечиваем доступ к модулю sp63_materials
if '.' not in sys.path:
    sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

from sp63_materials import (
    Concrete, Rebar, 
    list_concrete_grades, list_rebar_grades,
    generate_code_snippet
)

print("Модуль sp63_materials успешно загружен.")

Модуль sp63_materials успешно загружен.


## 1. Интерактивный дашборд выбора материалов и параметров

Выберите класс бетона, арматуры и условия эксплуатации. Графики деформационных диаграмм и сводная спецификация обновляются в реальном времени.

In [2]:
# Создание элементов управления (ipywidgets)
w_concrete_grade = widgets.Dropdown(
    options=list_concrete_grades(),
    value='B25',
    description='Класс бетона:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_humidity = widgets.Dropdown(
    options=[('Нормальная (40 - 75 %)', '40-75%'), ('Повышенная (> 75 %)', '>75%'), ('Сухая (< 40 %)', '<40%')],
    value='40-75%',
    description='Влажность среды:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_long_term = widgets.Checkbox(
    value=True,
    description='Длительное действие нагрузки (учет gamma_b1 = 0.90 и ползучести)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%')
)

w_vertical_casting = widgets.Checkbox(
    value=False,
    description='Вертикальное бетонирование h > 1.5 м (gamma_b3 = 0.85 для колонн/стоек)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%')
)

w_rebar_grade = widgets.Dropdown(
    options=list_rebar_grades(),
    value='A500',
    description='Класс арматуры:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

w_diag_model = widgets.RadioButtons(
    options=[
        ('Двухлинейная (п. 6.1.21)', 'bilinear'),
        ('Трехлинейная (п. 6.1.20)', 'trilinear'),
        ('Криволинейная нелинейная (Приложение Г)', 'nonlinear')
    ],
    value='nonlinear',
    description='Модель бетона:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='95%')
)

out_panel = widgets.Output()

def update_view(*args):
    with out_panel:
        clear_output(wait=True)
        
        # Инициализация объектов материалов
        gamma_b3_val = 0.85 if w_vertical_casting.value else 1.0
        concrete = Concrete(
            grade=w_concrete_grade.value,
            humidity=w_humidity.value,
            long_term=w_long_term.value,
            gamma_b3=gamma_b3_val
        )
        rebar = Rebar(
            grade=w_rebar_grade.value,
            long_term=w_long_term.value
        )
        
        # Построение графиков
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.8), dpi=110)
        plt.subplots_adjust(wspace=0.28)
        
        # 1. Сжатие бетона
        eps_c, sig_c = concrete.get_diagram(model=w_diag_model.value, state='compression', n_points=120)
        ax1.plot(eps_c * 1000, sig_c, color='#1f77b4', lw=2.2, label=f'{concrete.grade} ({w_diag_model.value})')
        ax1.axhline(concrete.Rb, color='#d62728', linestyle='--', alpha=0.7, label=f'Rb = {concrete.Rb} МПа')
        ax1.scatter([concrete.eps_b0 * 1000], [concrete.Rb], color='#d62728', zorder=5)
        ax1.annotate(f'({concrete.eps_b0*1000:.2f}‰; {concrete.Rb})',
                     xy=(concrete.eps_b0 * 1000, concrete.Rb),
                     xytext=(concrete.eps_b0 * 1000 * 0.65, concrete.Rb * 0.75),
                     arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.2),
                     fontsize=9, fontweight='bold', color='#d62728')
        ax1.set_title('Бетон при осевом сжатии', fontsize=12, fontweight='bold', pad=10)
        ax1.set_xlabel('Деформация укорочения $\\varepsilon_b$, ‰', fontsize=10)
        ax1.set_ylabel('Напряжение $\\sigma_b$, МПа', fontsize=10)
        ax1.grid(True, linestyle=':', alpha=0.6)
        ax1.legend(loc='lower right', fontsize=9)
        
        # 2. Растяжение бетона
        eps_t, sig_t = concrete.get_diagram(model='bilinear' if w_diag_model.value == 'bilinear' else 'trilinear', state='tension', n_points=120)
        ax2.plot(eps_t * 1000, sig_t, color='#2ca02c', lw=2.2, label=f'{concrete.grade} растяжение')
        ax2.axhline(concrete.Rbt, color='#d62728', linestyle='--', alpha=0.7, label=f'Rbt = {concrete.Rbt} МПа')
        ax2.scatter([concrete.eps_bt0 * 1000], [concrete.Rbt], color='#d62728', zorder=5)
        ax2.set_title('Бетон при осевом растяжении', fontsize=12, fontweight='bold', pad=10)
        ax2.set_xlabel('Деформация удлинения $\\varepsilon_{bt}$, ‰', fontsize=10)
        ax2.set_ylabel('Напряжение $\\sigma_{bt}$, МПа', fontsize=10)
        ax2.grid(True, linestyle=':', alpha=0.6)
        ax2.legend(loc='lower right', fontsize=9)
        
        # 3. Арматура при растяжении и сжатии
        eps_s, sig_s = rebar.get_diagram(state='tension', n_points=120)
        ax3.plot(eps_s * 1000, sig_s, color='#ff7f0e', lw=2.2, label=f'{rebar.grade} (Rs = {rebar.Rs} МПа)')
        ax3.axhline(rebar.Rs, color='#d62728', linestyle='--', alpha=0.6)
        ax3.scatter([rebar.eps_s0 * 1000], [rebar.Rs], color='#d62728', zorder=5)
        ax3.annotate(f'Текучесть ({rebar.eps_s0*1000:.2f}‰; {rebar.Rs} МПа)',
                     xy=(rebar.eps_s0 * 1000, rebar.Rs),
                     xytext=(rebar.eps_s0 * 1000 + 1.5, rebar.Rs * 0.75),
                     arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.2),
                     fontsize=9, fontweight='bold', color='#d62728')
        ax3.set_title(f'Арматура {rebar.grade} (Диаграмма Прандтля)', fontsize=12, fontweight='bold', pad=10)
        ax3.set_xlabel('Деформация удлинения $\\varepsilon_s$, ‰', fontsize=10)
        ax3.set_ylabel('Напряжение $\\sigma_s$, МПа', fontsize=10)
        ax3.grid(True, linestyle=':', alpha=0.6)
        ax3.legend(loc='lower right', fontsize=9)
        
        plt.show()
        
        # Сводные таблицы характеристик
        import html
        display(HTML(concrete.to_html()))
        display(HTML(rebar.to_html()))
        
        # Генерация сниппета кода
        snippet = generate_code_snippet(concrete, rebar)
        snippet_html = f'''<div style="margin-top: 20px;">\n<h3 style="margin-bottom: 8px;">Готовый фрагмент кода для вставки в расчетный блокнот:</h3>\n<pre style="background: #f8f9fa; border: 1px solid #dee2e6; border-radius: 6px; padding: 14px; font-family: Consolas, \'Fira Code\', Monaco, monospace; font-size: 13px; line-height: 1.45; overflow-x: auto; color: #212529;"><code>{html.escape(snippet)}</code></pre>\n</div>'''
        display(HTML(snippet_html))

# Привязка событий к виджетам
for w in [w_concrete_grade, w_humidity, w_long_term, w_vertical_casting, w_rebar_grade, w_diag_model]:
    w.observe(update_view, names='value')

# Разметка интерфейса
box_concrete = widgets.VBox([
    widgets.HTML("<b>Параметры бетона:</b>"),
    w_concrete_grade, w_humidity, w_diag_model, w_long_term, w_vertical_casting
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', margin='4px', border_radius='6px'))

box_rebar = widgets.VBox([
    widgets.HTML("<b>Параметры арматуры:</b>"),
    w_rebar_grade
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', margin='4px', border_radius='6px'))

ui_controls = widgets.HBox([box_concrete, box_rebar], layout=widgets.Layout(width='100%'))
display(ui_controls)
display(out_panel)

# Первоначальный рендер
update_view()

Output()

## 2. Пример прямого программного использования

Ниже показано, как вы можете вызывать библиотеку `sp63_materials` напрямую в любом расчетном скрипте или функции расчета сечения без графического интерфейса:

In [3]:
# Пример прямого расчета несущей способности прямоугольного сечения по прочности (п. 8.1.8 СП 63)
c = Concrete('B25', long_term=True)
s = Rebar('A500')

# Геометрия сечения (мм)
b = 300.0   # ширина балки, мм
h = 600.0   # высота балки, мм
a = 40.0    # защитный слой до центра арматуры, мм
h0 = h - a  # полезная высота, мм

# Площадь растянутой арматуры (3 стержня d25)
As = 3 * (np.pi * 25**2 / 4)  # ~1472 мм2

# 1. Высота сжатой зоны бетона x:
# Rs * As = Rb * b * x  =>  x = (Rs * As) / (Rb * b)
x = (s.Rs * As) / (c.Rb * b)

# Граничная относительная высота сжатой зоны xi_R (п. 8.1.9):
eps_s_el = s.Rs / s.Es
xi_R = 0.8 / (1.0 + eps_s_el / 0.0035)
x_R = xi_R * h0

print(f"Высота сжатой зоны x = {x:.1f} мм (предельная x_R = {x_R:.1f} мм, x/h0 = {x/h0:.3f} <= {xi_R:.3f})")

# 2. Предельный изгибающий момент Mult:
Mult = c.Rb * b * x * (h0 - 0.5 * x) / 1e6  # кН*м
print(f"Несущая способность по изгибающему моменту: Mult = {Mult:.2f} кН*м")

Высота сжатой зоны x = 163.6 мм (предельная x_R = 276.3 мм, x/h0 = 0.292 <= 0.493)
Несущая способность по изгибающему моменту: Mult = 306.32 кН*м
